# ML Exploration
Use this notebook to experiment with models before moving code to `model/train.py`.

In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier

## 1. Load processed data

In [25]:
TICKER = 'AAPL'
df = pd.read_csv(f'../data/processed/{TICKER}.csv', parse_dates=['Date'])

# Features: all engineered columns — raw prices excluded (non-stationary)
FEATURES = [
    'Return', 'MA5', 'MA20', 'Volume_Change',
    'RSI', 'MACD', 'MACD_Signal', 'BB_Upper', 'BB_Lower',
    'Return_Lag1', 'Return_Lag2'
]

X = df[FEATURES]
y = df['Target']

print(f"Dataset: {len(df)} rows | {df['Date'].min().date()} → {df['Date'].max().date()}")
print(f"Features: {FEATURES}")
print(f"\nTarget balance:\n{y.value_counts()}")

Dataset: 1217 rows | 2020-05-05 → 2025-03-07
Features: ['Return', 'MA5', 'MA20', 'Volume_Change', 'RSI', 'MACD', 'MACD_Signal', 'BB_Upper', 'BB_Lower', 'Return_Lag1', 'Return_Lag2']

Target balance:
Target
1    649
0    568
Name: count, dtype: int64


## 2. Train / test split

In [26]:
# --- Train / Test Split (70/30) ---
# We use a 70/30 split instead of the common 80/20 because stock data is limited
# (only ~5 years). A larger test set gives us a more reliable evaluation of how
# the model performs on unseen future data.
# IMPORTANT: shuffle=False is implicit here — we slice by index position to
# preserve chronological order. Training always on past, testing always on future.
split_idx = int(len(X) * 0.70)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# --- Feature Scaling ---
# StandardScaler standardizes each feature to zero mean and unit variance.
# This is critical for Logistic Regression (sensitive to feature scale) and
# also helps Gradient Boosting converge faster. We fit ONLY on training data
# to avoid data leakage — the test set is transformed using training statistics.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {len(X_train)} rows | Test: {len(X_test)} rows")

# --- TimeSeriesSplit Cross-Validation ---
# Standard k-fold CV randomly shuffles data, which causes future data to leak
# into training. TimeSeriesSplit always trains on the past and validates on the
# future — exactly how the model will be used in production.
# With 5 folds, we get 5 forward-rolling train→future evaluations.
tscv = TimeSeriesSplit(n_splits=5)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
    # XGBoost: a highly optimised gradient boosting implementation, often the
    # best performer on tabular data. eval_metric='logloss' suppresses warnings.
    'XGBoost':             XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss'),
}

# Train each model across all folds and record the average accuracy
cv_results = {}
for name, model in models.items():
    fold_scores = []
    for train_idx, val_idx in tscv.split(X_train_scaled):
        X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        model.fit(X_fold_train, y_fold_train)
        fold_scores.append(model.score(X_fold_val, y_fold_val))
    cv_results[name] = np.mean(fold_scores)
    print(f"{name}: avg CV accuracy = {np.mean(fold_scores):.4f}")

# Pick the model with the highest average cross-validation accuracy
best_model_name = max(cv_results, key=cv_results.get)
print(f"\nBest model: {best_model_name} ({cv_results[best_model_name]:.4f})")

Train: 851 rows | Test: 366 rows
Logistic Regression: avg CV accuracy = 0.5121
Random Forest: avg CV accuracy = 0.4596
Gradient Boosting: avg CV accuracy = 0.4922
XGBoost: avg CV accuracy = 0.4865

Best model: Logistic Regression (0.5121)


## 3. Model selection & evaluation

In [27]:
# TODO: fit model, print classification_report, plot confusion matrix